In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import os

In [ ]:
rave_sim_dir = Path('/mnt/d/rave-sim-main/rave-sim-main')
simulations_dir = Path('/mnt/d/rave-sim-main/rave-sim-main/output')
scratch_dir = simulations_dir

In [ ]:
sys.path.insert(0, str(rave_sim_dir / "big-wave"))
#print(str(rave_sim_dir / "big-wave"))
import multisim
import config
import util
import propagation

In [ ]:
densityAl=[2.68,2.69,2.7,2.71,2.72]
for ii in densityAl:
    config_dict = {
        "sim_params": {
            "N": 2**24,
            "dx":propagation.max_dx(0.8, 3e-7, 2**27, propagation.convert_energy_wavelength(21000)),
            "z_detector":101,
            "detector_size": 3e-3,
            "detector_pixel_size_x": 2e-5,
            "detector_pixel_size_y": 1,
            "chunk_size": 256 * 1024 * 1024 // 16,  # use 256MB chunks
        },
        "use_disk_vector": False,
        "save_final_u_vectors": False,
        "dtype": "c8",
        "multisource": {
            "type": "points",
            "energy_range": [21000-25, 21000+25],
            "x_range": [-2e-6, 2e-6],
            "z": 0.0,
            "nr_source_points": 5,
            "seed": 1,
            #"spectrum": "/mnt/d/rave-sim-main/rave-sim-main/spectrum/spectrum_25keV.h5",
        },
        "elements": [
            {
                "type": "sample",
                "z_start": 100,
                "pixel_size_x": 1 * 1e-6,
                "pixel_size_z": 1 * 1e-6,
                "grid_path":"/mnt/d/rave-sim-main/rave-sim-main/grid/Alstair.npy",
                "materials": [["Al", ii]],
                "x_positions":[0],
            },
        ],
    }
    print("dx: ", config_dict["sim_params"]["dx"])
    print("N: ", config_dict["sim_params"]["N"])
    sim_path = multisim.setup_simulation(config_dict, Path("."), simulations_dir)
    computed = config.load(Path(sim_path / 'computed.yaml'))
    
    #print("cutoff angles:", computed['cutoff_angles'])
    #print("source points:", computed['source_points'])
    for i in tqdm(range(config_dict["multisource"]["nr_source_points"])):
        os.system(f"CUDA_VISIBLE_DEVICES=0 /mnt/d/rave-sim-main/rave-sim-main/fast-wave/build-Release/fastwave -s {i} {sim_path}")
    wavefronts = util.load_wavefronts_filtered(sim_path, x_range=(-3e-6, 3e-6))
    print("nr sources loaded:", len(wavefronts))
    wavef= [result[0] for result in wavefronts]
    wf = np.sum(wavef, axis=0)
    print("nr phase steps:", wf.shape[0])
    print("nr detector pixels:", wf.shape[1])
    sp = config_dict["sim_params"]
    detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
    plt.plot(wf[0])
    from contextlib import redirect_stdout
    with open('output_Alstair_density_'+str(ii)+'.txt', 'w') as file:
        with redirect_stdout(file):
            for i in range(len(wf[0])):
                print(wf[0][i])
    print(detector_x)

In [ ]:
# Run this in a for loop to simulate all source points or
# alternatively run the source points as individual euler
# jobs
#for i in range(20):
#    multisim.run_single_simulation(sim_path, i, scratch_dir, save_keypoints_path=scratch_dir)


In [ ]:
wavefronts = util.load_wavefronts_filtered(sim_path, x_range=(-3e-6, 3e-6))
print("nr sources loaded:", len(wavefronts))
wavef= [result[0] for result in wavefronts]
wf = np.sum(wavef, axis=0)
print("nr phase steps:", wf.shape[0])
print("nr detector pixels:", wf.shape[1])

In [ ]:
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
from contextlib import redirect_stdout
with open('output_Alstair.txt', 'w') as file:
    with redirect_stdout(file):
        for i in range(len(wf[0])):
            print(wf[0][i])
print(detector_x)

In [ ]:
#以下是历史仿真结果

In [ ]:
#double_ring_911_844_803_0d_1e-7.npy
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
from contextlib import redirect_stdout
with open('output_3dc_1e7.txt', 'w') as file:
    with redirect_stdout(file):
        for i in range(len(wf[0])):
            print(wf[0][i])
print(detector_x)

In [ ]:
#double_ring_911_844_803_0d_1e-7.npy
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
from contextlib import redirect_stdout
with open('output_3dc_1e7.txt', 'w') as file:
    with redirect_stdout(file):
        for i in range(len(wf[0])):
            print(wf[0][i])
print(detector_x)

In [ ]:
#double_ring_911_844_803_0d_3e-7.npy

sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
from contextlib import redirect_stdout
with open('output_0d.txt', 'w') as file:
    with redirect_stdout(file):
        for i in range(len(wf[0])):
            print(wf[0][i])
print(detector_x)

In [ ]:
#double_ring_911_844_803_3d_3e-7.npy
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
from contextlib import redirect_stdout
with open('output_3d.txt', 'w') as file:
    with redirect_stdout(file):
        for i in range(len(wf[0])):
            print(wf[0][i])
print(detector_x)

In [ ]:
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
from contextlib import redirect_stdout
with open('output.txt', 'w') as file:
    with redirect_stdout(file):
        for i in range(len(wf[0])):
            print(wf[0][i])
print(detector_x)

In [ ]:
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
print(wf[0])
print(detector_x)

In [ ]:
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
print(wf[0])
print(detector_x)

## History

To see the interference pattern in empty space, we can record slices throughout the simulation and then plot them. `run_single_simulation` takes an optional argument `history_dz` defining the resolution with which we record the history.

Note that the history is not necessarily recorded with a constant z-spacing. Inside gratings and samples, one slice is recorded for every step. The history also records a list of z-coordinates at which the slices were recorded, which we can use for plotting.

In [ ]:
multisim.run_single_simulation(sim_path, 1, scratch_dir, save_keypoints_path=None, history_dz=0.02)

In [ ]:
# Path to the directory for the source with index 1
source_dir = multisim.get_sub_dir(sim_path, 1)

hist_x = np.load(source_dir / "history_x.npy")
hist_z = np.load(source_dir / "history_z.npy")
hist = np.load(source_dir / "history.npy")
plt.pcolormesh(
    hist_z,
    hist_x,
    hist,
    cmap="Greys_r",
    vmin=0,
    vmax=1e-6,
    shading="nearest",
)
plt.xlabel("z (m)")
plt.ylabel("x (m)")